## Instalar dependencias e importar librerias

In [3]:
try:
    import google.colab  # type: ignore
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    !pip install ultralytics opencv-python roboflow
else:
    print("Ejecucion local detectada. Instala dependencias en tu entorno antes de continuar.")
    print("Sugerencia: uv add ultralytics opencv-python roboflow jupyter ipykernel")

Ejecucion local detectada. Instala dependencias en tu entorno antes de continuar.
Sugerencia: uv add ultralytics opencv-python roboflow jupyter ipykernel


In [4]:
import os
import cv2
import torch
import shutil
from pathlib import Path
from ultralytics import YOLO

# Cambia solo estas variables si quieres reutilizar el notebook en otro entorno.
LOCAL_PROJECT_ROOT = Path("/home/fr4mes/unab/clase-ciencia-de-datos")
LOCAL_DATASET_DIR = LOCAL_PROJECT_ROOT / "dataset"
COLAB_DATASET_DIR = Path("/content/dataset")
DOWNLOADED_DATASET_NAME = "ppe-factory-1"
MODEL_NAME = "yolov8n.pt"

DATASET_DIR = COLAB_DATASET_DIR if IS_COLAB else LOCAL_DATASET_DIR
DATA_YAML = DATASET_DIR / "data.yaml"
TEST_IMAGES_DIR = DATASET_DIR / "test" / "images"
DOWNLOAD_DIR = (Path("/content") if IS_COLAB else Path.cwd()) / DOWNLOADED_DATASET_NAME

print(f"IS_COLAB: {IS_COLAB}")
print(f"DATASET_DIR: {DATASET_DIR}")
print(f"DOWNLOAD_DIR: {DOWNLOAD_DIR}")
print(f"DATA_YAML: {DATA_YAML}")

(null): No such file or directory
(null): No such file or directory


IS_COLAB: False
DATASET_DIR: /home/fr4mes/unab/clase-ciencia-de-datos/dataset
DOWNLOAD_DIR: /home/fr4mes/unab/ppe_detector_app/ppe-factory-1
DATA_YAML: /home/fr4mes/unab/clase-ciencia-de-datos/dataset/data.yaml


## Descargar dataset con RoboFlow

In [6]:
from roboflow import Roboflow

rf = Roboflow(api_key="dSMfDD4uPaMCKEoGOP5q")
project = rf.workspace("cicatriz").project("ppe-factory-bmdcj-alnpk")
version = project.version(1)
dataset = version.download("yolov8")

print(f"Dataset descargado en: {DOWNLOAD_DIR}")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to ppe-factory-1 in yolov8:: 100%|████████████████████| 21608/21608 [00:01<00:00, 17316.58it/s]

Dataset descargado en: /home/fr4mes/unab/ppe_detector_app/ppe-factory-1


## Mover dataset a la ruta configurada

In [7]:
if not DOWNLOAD_DIR.exists():
    raise FileNotFoundError(f"No se encontró el dataset descargado en {DOWNLOAD_DIR}")

if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)

shutil.move(str(DOWNLOAD_DIR), str(DATASET_DIR))
print(f"Dataset movido a: {DATASET_DIR}")

Dataset movido a: /home/fr4mes/unab/clase-ciencia-de-datos/dataset


## 4. Revisar estructura y etiquetas

Antes de entrenar, comprobamos que el dataset tenga la estructura correcta y que las etiquetas `.txt` existan en `train`, `valid` y `test`.

In [8]:
subdirs = ["train/images", "train/labels", "valid/images", "valid/labels", "test/images", "test/labels"]

print("=== Verificación de carpetas ===")
for subdir in subdirs:
    full_path = DATASET_DIR / subdir
    if not full_path.exists():
        print(f"ERROR: No se encontró {full_path}")
    else:
        print(f"OK: {full_path} contiene {len(os.listdir(full_path))} archivos")

print("\n=== Verificación de etiquetas ===")
for split in ["train", "valid", "test"]:
    labels_path = DATASET_DIR / split / "labels"
    if labels_path.exists():
        label_files = [f for f in os.listdir(labels_path) if f.endswith(".txt")]
        print(f"{split}: {len(label_files)} archivos de etiquetas")
    else:
        print(f"{split}: carpeta de etiquetas no encontrada")

=== Verificación de carpetas ===
OK: /home/fr4mes/unab/clase-ciencia-de-datos/dataset/train/images contiene 9770 archivos
OK: /home/fr4mes/unab/clase-ciencia-de-datos/dataset/train/labels contiene 9770 archivos
OK: /home/fr4mes/unab/clase-ciencia-de-datos/dataset/valid/images contiene 742 archivos
OK: /home/fr4mes/unab/clase-ciencia-de-datos/dataset/valid/labels contiene 742 archivos
OK: /home/fr4mes/unab/clase-ciencia-de-datos/dataset/test/images contiene 286 archivos
OK: /home/fr4mes/unab/clase-ciencia-de-datos/dataset/test/labels contiene 286 archivos

=== Verificación de etiquetas ===
train: 9770 archivos de etiquetas
valid: 742 archivos de etiquetas
test: 286 archivos de etiquetas


## Cargar modelo

In [10]:
model = YOLO(MODEL_NAME)

## Entrenar

In [11]:
# Modo rapido para probar el pipeline sin esperar tanto.
FAST_MODE = True

if FAST_MODE:
    EPOCHS = 20
    IMGSZ = 512
    BATCH = 8
    WORKERS = 2
    CACHE = True
    PATIENCE = 5
else:
    EPOCHS = 50
    IMGSZ = 640
    BATCH = 16
    WORKERS = 2
    CACHE = False
    PATIENCE = 10

print(f"FAST_MODE: {FAST_MODE}")
print(f"EPOCHS: {EPOCHS}, IMGSZ: {IMGSZ}, BATCH: {BATCH}, WORKERS: {WORKERS}, CACHE: {CACHE}, PATIENCE: {PATIENCE}")

FAST_MODE: True
EPOCHS: 20, IMGSZ: 512, BATCH: 8, WORKERS: 2, CACHE: True, PATIENCE: 5


In [ ]:
if not DATA_YAML.exists():
    raise FileNotFoundError(f"No se encontró el archivo {DATA_YAML}")

model.train(data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, workers=WORKERS, cache=CACHE, patience=PATIENCE)

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.13.0.dev20260422+rocm7.2 CUDA:0 (AMD Radeon Graphics, 16368MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/fr4mes/unab/clase-ciencia-de-datos/dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-3, nbs=64, nms=False, opset=None, optimize=F

(null): No such file or directory
(null): No such file or directory


Freezing layer 'model.22.dfl.conv.weight'
AMP: running Automatic Mixed Precision (AMP) checks...


## Probar imagen

In [ ]:
test_images = [f for f in os.listdir(TEST_IMAGES_DIR) if f.lower().endswith((".jpg", ".jpeg", ".png"))]

if not test_images:
    raise FileNotFoundError(f"No se encontraron imágenes en {TEST_IMAGES_DIR}")

image_path = TEST_IMAGES_DIR / test_images[0]
print(f"Probando con: {image_path}")
results = model.predict(str(image_path), save=True, imgsz=640)
results[0].show()

## Validar

In [ ]:
metrics = model.val()
print(metrics)

## Exportar

In [ ]:
model.export(format="onnx")
model.export(format="torchscript")
model.export(format="tflite")